# FSE Puglia redesign – survey analysis
Small sample (58 responses collected, 48 analysed: Puglia 25, Emilia-Romagna 23): everything here is **descriptive and directional**, not statistically conclusive.

Sections: 0 Setup · 1 Sample profile · 2 Adoption · 3 Usability (Likert) · 4 Dependence on others · 5 Failures & channels · 6 Time & anxiety · 7 Accessibility · 8 Open-text coding · 9 Export

## 0. Setup & cleaning

In [14]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import re, os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

PATH = "fse_answers.csv"
raw = pd.read_csv(PATH)

# Rename the long questions to short names (by position, so the order of the form must not change)
short = ["timestamp","age","region","digital_skill","difficulty",
         "used_fse","experience","frequency","top_service","useful_features",
         "autonomy","failed_op","failed_why","non_use_why",
         "booking_channel","non_online_why",
         "l_easy","l_confident","l_clear_steps","l_icons","l_info","anxiety",
         "booking_time","guided_someone","guided_why","guide_ease","helped_alone_next",
         "booked_for_other","booked_for_why","satisfaction","wishes"]
assert len(short) == raw.shape[1]
df = raw.copy(); df.columns = short
df["id"] = df.index

# Cleaning
# is_string_dtype also catches pandas>=3 "str" columns (there dtype == "object" is False and nothing was stripped)
df = df.apply(lambda s: s.str.strip() if pd.api.types.is_string_dtype(s) else s)
# '< 60 anni' was a typo in the form options: it means '> 60 anni' (confirmed with the form authors)
df["age"] = df["age"].replace({"< 60 anni": "> 60 anni"})

# Keep only the two regions under comparison; the other 10 respondents ("Altro" 5, "Veneto" 5) stay in df_all as reference
df_all = df.copy()
df = df[df["region"].isin(["Puglia", "Emilia-Romagna"])].copy()
print(df["region"].value_counts())
print(df["age"].value_counts())

likert = ["l_easy","l_confident","l_clear_steps","l_icons","l_info"]
time_order = ["meno di 5 minuti","5-10 minuti","10-20","non riesco a completare la prenotazione"]
df["booking_time"] = pd.Categorical(df["booking_time"], time_order, ordered=True)

def explode_multi(col):
    # multi-select answers are separated by ';'
    out = (df[["id","region",col]].dropna().assign(**{col: lambda d: d[col].str.split(";")})
           .explode(col).reset_index(drop=True))
    out[col] = out[col].str.strip()
    return out[out[col] != ""]

def save_table_png(table, filename, title=None):
    """Render a small DataFrame as a clean PNG image in outputs/."""
    t = table.reset_index()
    t.columns = [str(c) for c in t.columns]
    cells = t.astype(str).values.tolist()
    widths = [max([len(c)] + [len(r[j]) for r in cells]) for j, c in enumerate(t.columns)]
    fig, ax = plt.subplots(figsize=(max(4, 0.11 * sum(widths) + 0.6), 0.4 * (len(t) + 1) + (0.7 if title else 0.3)))
    ax.axis("off")
    tb = ax.table(cellText=cells, colLabels=list(t.columns), cellLoc="center", loc="upper center")
    tb.auto_set_font_size(False); tb.set_fontsize(10)
    tb.auto_set_column_width(list(range(len(t.columns))))
    for (r, c), cell in tb.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if r == 0: cell.set_facecolor("#e8e8e8"); cell.set_text_props(weight="bold")
        if c == 0 and r > 0: cell.set_text_props(ha="left"); cell._loc = "left"
    if title: ax.set_title(title, fontsize=11, weight="bold")
    path = f"outputs/{filename}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)
    # trim leftover white space around the table
    from PIL import Image, ImageChops
    im = Image.open(path).convert("RGB")
    box = ImageChops.difference(im, Image.new("RGB", im.size, "white")).getbbox()
    if box:
        pad = 20
        im.crop((max(box[0]-pad, 0), max(box[1]-pad, 0), min(box[2]+pad, im.width), min(box[3]+pad, im.height))).save(path)

def crosstab_sorted(rows, cols):
    # counts by group + total column, largest first (the old margins=True put the "All" row on top after sorting)
    t = pd.crosstab(rows, cols)
    t["All"] = t.sum(axis=1)
    return t.sort_values("All", ascending=False)

# Main comparison groups. Puglia = client, Emilia-Romagna = competitor (other regions are only kept in df_all)
GROUPS = ["Puglia","Emilia-Romagna"]
df.shape

region
Puglia            25
Emilia-Romagna    23
Name: count, dtype: int64
age
18 - 35 anni    20
36 - 60 anni    16
> 60 anni       12
Name: count, dtype: int64


(48, 32)

### Data-quality checks
Answers that do not match the form's routing (probably the follow-up questions were not conditional in the form). They are **kept** in the analysis; read the related tables (sections 3, 4, 6) with this in mind.

In [15]:
dq = {
    "Said they never guided anyone, but rated how easy it was": ((df.guided_someone == "No") & df.guide_ease.notna()).sum(),
    "Said they never guided anyone, but answered 'did the person manage alone next time'": ((df.guided_someone == "No") & df.helped_alone_next.notna()).sum(),
    "Said they never booked for others, but gave a reason": ((df.booked_for_other == "No") & df.booked_for_why.notna()).sum(),
    "Non-users of the FSE who rated satisfaction": ((df.used_fse == "No") & df.satisfaction.notna()).sum(),
    "Non-users of the FSE who answered the Likert block": ((df.used_fse == "No") & df.l_easy.notna()).sum(),
}
display(pd.Series(dq, name="n respondents").to_frame())

,n respondents
"Said they never guided anyone, but rated how easy it was",6
"Said they never guided anyone, but answered 'did the person manage alone next time'",5
"Said they never booked for others, but gave a reason",2
Non-users of the FSE who rated satisfaction,7
Non-users of the FSE who answered the Likert block,8


## 1. Sample profile
Check whether the regions are comparable before comparing scores.

In [16]:
for col in ["age","digital_skill"]:
    print(f"\n== {col} ==")
    print(pd.crosstab(df[col], df["region"], margins=True))


== age ==
region        Emilia-Romagna  Puglia  All
age                                      
18 - 35 anni               8      12   20
36 - 60 anni               8       8   16
> 60 anni                  7       5   12
All                       23      25   48

== digital_skill ==
region         Emilia-Romagna  Puglia  All
digital_skill                             
2                           1       0    1
3                           9      13   22
4                           7       6   13
5                           6       6   12
All                        23      25   48


In [17]:
age_pct   = pd.crosstab(df["region"], df["age"], normalize="index").mul(100).round(1)
skill_pct = pd.crosstab(df["region"], df["digital_skill"], normalize="index").mul(100).round(1)
n_by_region = df["region"].value_counts()

AGE_ORDER = ["18 - 35 anni", "36 - 60 anni", "> 60 anni"]
AGE_COLORS = dict(zip(AGE_ORDER, ["#fdd0a2", "#fd8d3c", "#a63603"]))      # light -> dark orange

SKILL_ORDER = [1, 2, 3, 4, 5]
SKILL_COLORS = dict(zip(SKILL_ORDER, ["#c6dbef", "#9ecae1", "#6baed6", "#3182bd", "#08519c"]))  # light -> dark blue

# Region labels with n, e.g. "Puglia (n=25)"
labels = {r: f"{r} (n={n_by_region[r]})" for r in age_pct.index}

fig = make_subplots(rows=1, cols=2, subplot_titles=("Age by region (%)",
                                                    "Digital skill (1-5) by region (%)"))

for col in age_pct.columns:
    fig.add_trace(go.Bar(name=str(col), x=[labels[r] for r in age_pct.index], y=age_pct[col],
                         legendgroup="age", legendgrouptitle_text="Age",
                         marker_color=AGE_COLORS[col],
                         hovertemplate="%{x}<br>" + str(col) + ": %{y:.1f}%<extra></extra>"),
                  row=1, col=1)

for col in skill_pct.columns:
    fig.add_trace(go.Bar(name=f"skill {col}", x=[labels[r] for r in skill_pct.index], y=skill_pct[col],
                         legendgroup="skill", legendgrouptitle_text="Skill",
                         marker_color=SKILL_COLORS[col],
                         hovertemplate="%{x}<br>skill " + str(col) + ": %{y:.1f}%<extra></extra>"),
                  row=1, col=2)

fig.update_layout(barmode="stack", height=420, width=1000, template="simple_white",
                  legend=dict(font=dict(size=10)))
fig.update_yaxes(title_text="%", range=[0, 100])
fig.show()

## 2. Adoption funnel

In [18]:
# Group redundant free-text answers of top_service
TOP_SERVICE_GROUPS = {
    "Prescrizione medicinali o visite dal medico di famiglia": "Prescrizioni e ricette",
    "Richiesta prescrizione medicinali": "Prescrizioni e ricette",
    "Scaricare ricette da mostrare in farmacia per acquistare medicinali": "Prescrizioni e ricette",
}
df["top_service"] = df["top_service"].replace(TOP_SERVICE_GROUPS)

adopt = pd.crosstab(df["region"], df["used_fse"], margins=True)
adopt["% used"] = (adopt.get("Si",0) / adopt["All"] * 100).round(0)
display(adopt)
display(pd.crosstab(df["age"], df["used_fse"], margins=True))
display(pd.crosstab(df["frequency"].fillna("(non-user)"), df["region"]))
display(pd.crosstab(df["top_service"].fillna("(non-user)"), df["region"]))

used_fse,No,Si,All,% used
region,,,,
Emilia-Romagna,3,20,23,87.0
Puglia,9,16,25,64.0
All,12,36,48,75.0


used_fse,No,Si,All
age,,,
18 - 35 anni,6,14,20
36 - 60 anni,1,15,16
> 60 anni,5,7,12
All,12,36,48


region,Emilia-Romagna,Puglia
frequency,,
(non-user),3,9
Circa una volta al mese,7,2
Più volte al mese,3,1
Qualche volta all'anno,10,13


region,Emilia-Romagna,Puglia
top_service,,
(non-user),3,9
Consultazione diario vaccinazioni,0,3
Consultazione referti,14,7
Prenotazione di una visita,2,6
Prescrizioni e ricette,3,0
Stampa prescrizioni e referti,1,0


In [19]:
REGION_COLORS = {"Puglia": "#2a9d8f", "Emilia-Romagna": "#8e6bb0"}
steps = ["Respondents", "Have used the FSE", "Usually book visits online"]

fig = go.Figure()
for r in ["Puglia", "Emilia-Romagna"]:
    d = df[df["region"] == r]
    vals = [
        len(d),
        (d["used_fse"] == "Si").sum(),
        (d["booking_channel"] == "Tramite il portale online").sum(),
    ]
    pct = [v / vals[0] * 100 for v in vals]
    fig.add_trace(go.Funnel(
        name=f"{r} (n={vals[0]})", y=steps, x=pct,
        text=[f"{v} people ({p:.0f}%)" for v, p in zip(vals, pct)], textinfo="text",
        marker_color=REGION_COLORS[r],
        hovertemplate="%{y}<br>%{text}<extra></extra>",
    ))

fig.update_layout(funnelmode="group", height=450, width=950, template="simple_white",
                  title="Adoption funnel (% of respondents in each region)",
                  legend=dict(orientation="h", y=-0.1))
fig.show()

In [20]:
REASON_GROUPS = {
    "Non sapevo che esistesse": "Awareness: didn't know it exists",
    "Non so come accedere": "Access: don't know how to log in",
    "Lo trovo complicato": "Usability: find it complicated",
    "Preferisco altri metodi": "Preference / no need",
    "Non ne ho avuto la necessità": "Preference / no need",
    "Non c'è mai stata necessità": "Preference / no need",
}

nu = explode_multi("non_use_why")
nu["group"] = nu["non_use_why"].map(REASON_GROUPS).fillna("Other")
counts = pd.crosstab(nu["group"], nu["region"]).reindex(columns=["Puglia", "Emilia-Romagna"], fill_value=0)
counts = counts.loc[counts.sum(axis=1).sort_values().index]      # largest at the top

n_nonusers = df[df["used_fse"] == "No"].groupby("region").size()

fig = go.Figure()
for r in ["Puglia", "Emilia-Romagna"]:
    fig.add_trace(go.Bar(
        y=counts.index, x=counts[r], orientation="h",
        name=f"{r} ({n_nonusers.get(r, 0)} non-users)", marker_color=REGION_COLORS[r],
        text=counts[r], textposition="outside",
    ))
fig.update_layout(barmode="group", height=420, width=950, template="simple_white",
                  title="Reasons for not using the FSE (number of people, multiple answers allowed)",
                  xaxis_title="People", margin=dict(l=260), legend=dict(orientation="h", y=-0.2))
fig.show()

## 3. Usability scores (Likert 1-5), by region

In [21]:
REGION_COLORS = {"Puglia": "#2a9d8f", "Emilia-Romagna": "#8e6bb0"}

ITEMS = {                      # column: (label, which answers count as "good")
    "l_easy":        ("Booking is easy", "high"),
    "l_confident":   ("I feel confident, no fear of mistakes", "high"),
    "l_clear_steps": ("Steps are clear", "high"),
    "l_icons":       ("Buttons and icons are intuitive", "high"),
    "l_info":        ("Information is easy to understand", "high"),
    "anxiety":       ("Low anxiety while booking", "low"),
    "satisfaction":  ("Overall satisfaction", "high"),
}

rows = []
for col, (label, good) in ITEMS.items():
    for r in ["Puglia", "Emilia-Romagna"]:
        s = df.loc[df["region"] == r, col].dropna()
        share = (s >= 4).mean() if good == "high" else (s <= 2).mean()
        rows.append({"item": label, "region": r, "pct": round(share * 100), "n": len(s)})
plot_df = pd.DataFrame(rows)

fig = go.Figure()
for r in ["Puglia", "Emilia-Romagna"]:
    d = plot_df[plot_df["region"] == r]
    fig.add_trace(go.Bar(y=d["item"], x=d["pct"], orientation="h",
                         name=f"{r} (n={d['n'].min()}-{d['n'].max()})",
                         marker_color=REGION_COLORS[r],
                         text=d["pct"].astype(str) + "%", textposition="outside",
                         hovertemplate="%{y}<br>" + r + ": %{x}%<extra></extra>"))
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(range=[0, 100], title_text="% of respondents giving a positive answer")
fig.update_layout(barmode="group", height=480, width=900, template="simple_white",
                  margin=dict(l=260), legend=dict(orientation="h", y=-0.2),
                  title="Positive answers by region (higher = better for every row)")
fig.show()

## 4. Dependence on others

In [22]:
# Group the free-text reasons (section 4)
BOOKED_FOR_WHY_GROUPS = {
    "Età avanzata": "Età avanzata",
    "Difficoltà digitali": "Difficoltà / scarse competenze digitali",
    "Language barrier": "Barriere di accesso (lingua, SPID/CIE)",
    "Mancanza di identità elettronica (SPID/CIE)": "Barriere di accesso (lingua, SPID/CIE)",
}

GUIDED_WHY_GROUPS = {
    "Analfabetismo digitale": "Scarse competenze digitali / inesperienza",
    "La persona utilizza il servizio raramente ed ha scarse competenze digitali": "Scarse competenze digitali / inesperienza",
    "Per inesperienza": "Scarse competenze digitali / inesperienza",
    "Perché non si possedevano le competenze adatte.": "Scarse competenze digitali / inesperienza",
    "Perché non riuscivano a farlo": "Difficoltà nell'uso del servizio",
    "Non capivano": "Difficoltà nell'uso del servizio",
    "Persona anziana": "Età avanzata",
    "Perché non conosceva il servizio": "Altro (non conosceva il servizio, supporto in sportello)",
    "Perché offrivo supporto in uno sportello di facilitazione digitale": "Altro (non conosceva il servizio, supporto in sportello)",
}

df["booked_for_why"] = df["booked_for_why"].replace(BOOKED_FOR_WHY_GROUPS)
df["guided_why"] = df["guided_why"].replace(GUIDED_WHY_GROUPS)

display(pd.crosstab(df["autonomy"].fillna("(non-user)"), df["age"]))
display(pd.crosstab(df["autonomy"].fillna("(non-user)"), df["digital_skill"]))
print("\nGuided someone:"); display(pd.crosstab(df["guided_someone"], df["region"]))
print("\nHelped person managed alone next time:"); display(pd.crosstab(df["helped_alone_next"].dropna(), df["region"]))
print("\nBooked for a relative/friend:"); display(pd.crosstab(df["booked_for_other"].dropna(), df["region"]))
display(pd.crosstab(df["booked_for_why"], df["region"], margins=True))
display(pd.crosstab(df["guided_why"], df["region"], margins=True))

age,18 - 35 anni,36 - 60 anni,> 60 anni
autonomy,,,
(non-user),6,1,5
A volte con aiuto,0,5,1
In autonomia,14,10,6


digital_skill,2,3,4,5
autonomy,,,,
(non-user),0,6,3,3
A volte con aiuto,1,5,0,0
In autonomia,0,11,10,9



Guided someone:


region,Emilia-Romagna,Puglia
guided_someone,,
No,18,16
Si,5,8



Helped person managed alone next time:


region,Emilia-Romagna,Puglia
helped_alone_next,,
"No, mi è stato chiesto nuovamente aiuto",2,6
Si,5,6



Booked for a relative/friend:


region,Emilia-Romagna,Puglia
booked_for_other,,
No,17,15
Si,1,7


region,Emilia-Romagna,Puglia,All
booked_for_why,,,
"Barriere di accesso (lingua, SPID/CIE)",1,0,1
Difficoltà / scarse competenze digitali,0,3,3
Età avanzata,1,4,5
Language barrier,1,0,1
All,3,7,10


region,Emilia-Romagna,Puglia,All
guided_why,,,
"Altro (non conosceva il servizio, supporto in sportello)",2,0,2
Età avanzata,1,0,1
Non capivano,1,0,1
Per inesperienza,0,1,1
Perché non riuscivano a farlo,0,1,1
Scarse competenze digitali / inesperienza,0,3,3
All,4,5,9


## 5. Failure points & booking channels

In [23]:
display(df.loc[df.failed_why.notna(), ["region","failed_why"]])
print("Channel by region:"); display(pd.crosstab(df["booking_channel"], df["region"], margins=True))
nonline = explode_multi("non_online_why")
display(crosstab_sorted(nonline["non_online_why"], nonline["region"]))

,region,failed_why
0,Emilia-Romagna,Per prenotare un'ecografia non mi mostra le disponibilità e non mi da alternative. Quindi rimango nella pagina della...
18,Puglia,Mancanza di date disponibili
22,Emilia-Romagna,"Perché a volte nel tentativo di fare una prenotazione on line,il programma non va avanti ma continua sempre a ritorn..."
24,Emilia-Romagna,Non sono riuscito a trovare il libretto dei vaccini
29,Puglia,Bug del sito
33,Puglia,Si
42,Puglia,Per difficoltà personali con il sistema
50,Emilia-Romagna,"Non saprei, forse un po’ per mia ignoranza o un riscontro non chiaro"


Channel by region:


region,Emilia-Romagna,Puglia,All
booking_channel,,,
Di persona allo sportello,8,9,17
Tramite chiamata,7,9,16
Tramite il portale online,8,6,14
All,23,24,47


region,Emilia-Romagna,Puglia,All
non_online_why,,,
È più veloce di persona o per telefono,10,13,23
Non so come fare,3,3,6
Le disponibilità delle strutture non sono chiare sul portale,1,3,4
Ho difficoltà nell'utilizzo,1,1,2
Non mi fido,2,0,2
Grazie a Dio non ho bisogno di visite specialistiche,0,1,1
In alcuni casi è necessaria l'interazione,0,1,1
Mi sento più sicura al cup,1,0,1
Non L ho mai utilizzato,0,1,1


## 6. Time on task, anxiety, satisfaction

In [24]:
display(pd.crosstab(df["booking_time"], df["region"]))
print("Anxiety (1 = none, 5 = very high):"); display(pd.crosstab(df["anxiety"].astype("Int64"), df["region"]))
print("Satisfaction (1-5):"); display(pd.crosstab(df["satisfaction"].astype("Int64"), df["region"]))

region,Emilia-Romagna,Puglia
booking_time,,
meno di 5 minuti,4,4
5-10 minuti,6,7
10-20,5,6
non riesco a completare la prenotazione,1,1


Anxiety (1 = none, 5 = very high):


region,Emilia-Romagna,Puglia
anxiety,,
1,8,6
2,4,3
3,3,5
4,2,4
5,1,1


Satisfaction (1-5):


region,Emilia-Romagna,Puglia
satisfaction,,
1,1,0
2,4,4
3,4,8
4,6,8
5,6,0


## 7. Accessibility
Only 3 respondents report a difficulty (all Puglia, all FSE users): use as anecdotal evidence only.

In [25]:
display(df[df.difficulty=="Si"][["region","age","digital_skill","used_fse","satisfaction","experience"]])

,region,age,digital_skill,used_fse,satisfaction,experience
33,Puglia,36 - 60 anni,3,Si,3.0,NaN
38,Puglia,36 - 60 anni,3,Si,3.0,Sono riuscita a rilevare l esito degli esami dopo una donazione di sangue
42,Puglia,36 - 60 anni,3,Si,3.0,Lo ritengo utile


## 8. Open-text coding
Step 1: keyword-based first pass (fast, imperfect). Step 2: **export and review manually** — a human check is needed with such short, varied answers.

In [26]:
THEMES = {
  "availability / calendar":  r"disponibil|calendario|\bdate\b|posti|nessun risultato|sedi|struttur|vicin",
  "login / SPID / CIE":       r"spid|\bcie\b|accesso|accedere|credenzial|login|password",
  "bugs / errors / speed":    r"bug|non funziona|errore|lent|torna(?:re)? all|blocc",
  "clarity / simplicity":     r"chiar|semplic(?!emente)|intuitiv|complicat|confus",
  "reminders / notifications":r"promemoria|notific|avvis|sms|mail|ricorda",
  "app / mobile":             r"\bapp\b|applicazione|smartphone|mobile",
  "documents / reports":      r"referti|document|ricett|prescriz|vaccin|analisi|\bstori",
  "help / delegate":          r"delega|familiar|aiuto|assisten|anzian|guida|tutorial",
  "payment / tickets":        r"pagament|ticket",
}
def code_text(s):
    s = str(s).lower()
    return [t for t, pat in THEMES.items() if re.search(pat, s)]

text_cols = {"experience":"experience","useful_features":"useful_features","wishes":"wishes"}
coded = {}
theme_counts = {}
for name, col in text_cols.items():
    t = df[["id","region",col]].dropna()
    t = t[t[col].str.len() > 3].copy()
    t["themes"] = t[col].apply(code_text)
    coded[name] = t
    ex = t.explode("themes").dropna(subset=["themes"]).reset_index(drop=True)
    print(f"\n=== {name}: {len(t)} answers; theme counts by region ===")
    theme_counts[name] = crosstab_sorted(ex["themes"], ex["region"])
    display(theme_counts[name])
    print("Uncoded answers:", (t["themes"].str.len()==0).sum())


=== experience: 28 answers; theme counts by region ===


region,Emilia-Romagna,Puglia,All
themes,,,
clarity / simplicity,2,4,6
availability / calendar,1,3,4
documents / reports,2,0,2
login / SPID / CIE,1,0,1


Uncoded answers: 18

=== useful_features: 24 answers; theme counts by region ===


region,Emilia-Romagna,Puglia,All
themes,,,
documents / reports,10,3,13
login / SPID / CIE,1,0,1
reminders / notifications,1,0,1


Uncoded answers: 11

=== wishes: 20 answers; theme counts by region ===


region,Emilia-Romagna,Puglia,All
themes,,,
clarity / simplicity,4,5,9
availability / calendar,4,0,4
login / SPID / CIE,2,0,2
app / mobile,1,0,1


Uncoded answers: 7


In [27]:
# Wishes from Puglia vs competitor, side by side, for manual reading
pd.set_option("display.max_colwidth", 300)
display(coded["wishes"].sort_values("region")[["region","wishes","themes"]])

,region,wishes,themes
0,Emilia-Romagna,"Vorrei che durante la prenotazione di una visita medica mi venga mostrato un calendario delle disponibilità dell'AUSL della mia provincia. Mi sembra strano che mi dica ""Nessun risultato per i criteri di ricerca impostati. Premi ""Cambia azienda"" per cambiare azienda ed eseguire una nuova ricerca"".",[availability / calendar]
47,Emilia-Romagna,Per la mia esperienza non ho cambiamenti da chiedere,[]
34,Emilia-Romagna,Tempi più lunghi per completare l'operazione prima che si aggiorni la pagina e maggiore chiarezza relativa alle strutture con disponibilità di prenotazione,"[availability / calendar, clarity / simplicity]"
51,Emilia-Romagna,Nessuna,[]
22,Emilia-Romagna,Più semplicita ' nell ' accesso,"[login / SPID / CIE, clarity / simplicity]"
16,Emilia-Romagna,Migliore esperienza di utilizzo nel cercare e visualizzare le strutture disponibili e le fasce orarie,[availability / calendar]
24,Emilia-Romagna,"Miglior adattamento a schermo smartphone, icone troppo piccole, non esageratamente intuitive","[clarity / simplicity, app / mobile]"
9,Emilia-Romagna,Maggiore chiarezza,[clarity / simplicity]
7,Emilia-Romagna,Per stranieri che non abbiamo l’identità digitale ma pagano per avere il ssn possano accedere al fascicolo sanitario elettronico,[login / SPID / CIE]
5,Emilia-Romagna,"La possibilità di scorrere in un calendario di disponibilità, la visualizzazione delle sedi tramite mappa.",[availability / calendar]


## 9. Export tables for the report (PNG)

In [29]:
os.makedirs("outputs", exist_ok=True)
save_table_png(pd.crosstab(df["region"], df["age"]), "age_by_region", "Age by region (people)")
save_table_png(adopt, "adoption", "FSE adoption by region")
save_table_png(pd.crosstab(df["booking_channel"], df["region"]), "booking_channel", "Booking channel by region (people)")
for k, v in theme_counts.items():
    save_table_png(v, f"themes_{k}", f"Open-text themes: {k} (people)")
print("Saved PNG tables to outputs/")

Saved PNG tables to outputs/
